# 07 — Analyze Results

Creates summary tables, plots, and failure-case CSVs.

In [ ]:
from pathlib import Path
import os, random, time, gc, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

EXP_DIR = Path('/media/wadud/DriveUbuntu/GaitRecognition 2.0')
POSE_TAG = None  # None = auto-detect folder under data/poses with most .npz files
POSES_DIR = EXP_DIR / 'data' / 'poses'
SPLIT_DIR = EXP_DIR / 'data' / 'splits'
REPORT_DIR = EXP_DIR / 'data' / 'reports'
RESULT_DIR = EXP_DIR / 'results'
CHECKPOINT_DIR = EXP_DIR / 'checkpoints'
LOG_DIR = EXP_DIR / 'logs'
for d in [SPLIT_DIR, REPORT_DIR, RESULT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def detect_pose_root(poses_dir=POSES_DIR, pose_tag=POSE_TAG):
    if pose_tag is not None:
        root = poses_dir / pose_tag
        if not root.exists():
            raise FileNotFoundError(root)
        return root
    candidates = [p for p in poses_dir.iterdir() if p.is_dir()] if poses_dir.exists() else []
    if not candidates:
        raise FileNotFoundError(f'No pose folders under {poses_dir}')
    counts = sorted([(len(list(p.rglob('*.npz'))), p) for p in candidates], reverse=True, key=lambda x: x[0])
    return counts[0][1]

POSE_ROOT = detect_pose_root()
print('EXP_DIR  :', EXP_DIR)
print('POSE_ROOT:', POSE_ROOT)

SPLIT_NAME='LT'; PLOT_DIR=RESULT_DIR/'plots'; PLOT_DIR.mkdir(parents=True,exist_ok=True)
summary_csv=RESULT_DIR/f'rank1_{SPLIT_NAME}_summary.csv'; per_view_csv=RESULT_DIR/f'rank1_{SPLIT_NAME}_per_view.csv'; all_csv=RESULT_DIR/f'rank1_{SPLIT_NAME}_all_probe_results.csv'
for p in [summary_csv,per_view_csv,all_csv]: assert p.exists(),p
df_summary=pd.read_csv(summary_csv); df_per=pd.read_csv(per_view_csv); df_all=pd.read_csv(all_csv)
display(df_summary); display(df_per.head()); display(df_all.head())

In [ ]:
plt.figure(figsize=(6,4)); plt.bar(df_summary.condition,df_summary.rank1*100); plt.ylim(0,100); plt.ylabel('Rank-1 Accuracy (%)'); plt.xlabel('Condition'); plt.title(f'GaitTR CASIA-B {SPLIT_NAME}')
for i,r in df_summary.iterrows(): plt.text(i,r.rank1*100+1,f'{r.rank1*100:.2f}%',ha='center')
plt.grid(axis='y',alpha=0.3); plt.tight_layout(); out=PLOT_DIR/f'rank1_{SPLIT_NAME}_condition_bar.png'; plt.savefig(out,dpi=150); plt.show(); print('Saved:',out)

In [ ]:
plt.figure(figsize=(10,5))
for cond in sorted(df_per.condition.unique()):
    sub=df_per[df_per.condition==cond].copy(); sub['view_int']=sub.view.astype(int); sub=sub.sort_values('view_int')
    plt.plot(sub.view_int,sub.rank1*100,marker='o',label=cond)
plt.xlabel('Probe view'); plt.ylabel('Rank-1 Accuracy (%)'); plt.title(f'Per-view Accuracy {SPLIT_NAME}'); plt.ylim(0,100); plt.grid(True,alpha=0.3); plt.legend(); plt.tight_layout()
out=PLOT_DIR/f'rank1_{SPLIT_NAME}_per_view.png'; plt.savefig(out,dpi=150); plt.show(); print('Saved:',out)

In [ ]:
fail=df_all[df_all.correct==0].copy(); fail_csv=RESULT_DIR/f'failure_cases_{SPLIT_NAME}.csv'; fail.to_csv(fail_csv,index=False)
print('Total:',len(df_all),'Failed:',len(fail),'Saved:',fail_csv); display(fail.head(30))
fv=fail.groupby(['condition','view']).size().reset_index(name='num_failures'); fv.to_csv(RESULT_DIR/f'failure_by_view_{SPLIT_NAME}.csv',index=False); display(fv)
paper=df_summary.copy(); paper['rank1_percent']=(paper.rank1*100).round(2); paper=paper[['condition','rank1_percent','num_probe']]; paper.to_csv(RESULT_DIR/f'paper_table_rank1_{SPLIT_NAME}.csv',index=False); display(paper)